In [1]:
import sys
from pathlib import Path

# Aggiunge la radice del progetto ('sumo-stgcn') al sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch

# Importa trattando 'src' come package
from src.dataset import GraphBuilder, DetectorParser
from src.blocks.gcn import GCN

/mnt/external1/irene/sumo-stgcn/.venv/lib/python3.14/site-packages/torch/__config__.py:9: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._show_config()
/mnt/external1/irene/sumo-stgcn/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
net_path = "/mnt/external1/irene/sumo-stgcn/input/map_restricted.net.xml"
det_dir = "/mnt/external1/irene/sumo-stgcn/input/DetOut_Day"

In [3]:
# 1. Costruisci il grafo baseline (nessuna chiusura, severity=1 ovunque)
graph = GraphBuilder().build(net_path, tls_states={})
edge_id_to_idx = graph["edge_id_to_idx"]

# 2. Parsea i detector per avere x_dynamic reale (non serve per questo test, ma verifica che il parsing funzioni)
x_dynamic = DetectorParser(det_dir, edge_id_to_idx).parse()
print("x_dynamic shape:", x_dynamic.shape)  # [T, N_nodes, 3]

# 3. Prendi un singolo snapshot come input x (usa x_static + x_dynamic[0], adatta al tuo Snapshot.to_data())
x = torch.cat([graph["x_static"], x_dynamic[0]], dim=-1)  # [N_nodes, N_feat_tot] — ADATTA se il tuo schema concatena diversamente

model = GCN(hidden_dims=[x.shape[1], 32, 16], dropout_prob=0.0)
model.eval()

class DummyData:
    pass

data = DummyData()
data.x = x
data.edge_index = graph["edge_index"]
data.edge_attr = graph["edge_attr"]

with torch.no_grad():
    out_baseline = model(data)  # severity = 1 ovunque, come da edge_attr originale

# 4. Perturba severity su UN edge (scegli un connection_pair a caso, es. il primo)
data_perturbed = DummyData()
data_perturbed.x = x
data_perturbed.edge_index = graph["edge_index"]
data_perturbed.edge_attr = graph["edge_attr"].clone()
data_perturbed.edge_attr[0, -1] = 0.2  # riduci severity sul primo edge del grafo

with torch.no_grad():
    out_perturbed = model(data_perturbed)

diff = (out_baseline - out_perturbed).abs()
print("Differenza massima:", diff.max().item())
print("Nodi impattati (diff > 1e-4):", (diff.sum(dim=-1) > 1e-4).sum().item(), "/", diff.shape[0])

x_dynamic shape: torch.Size([289, 234, 3])
Differenza massima: 0.14617574214935303
Nodi impattati (diff > 1e-4): 3 / 234


In [5]:
perturbed_edge_idx = 0  # lo stesso che hai perturbato
src_node, dst_node = graph["edge_index"][:, perturbed_edge_idx].tolist()
print(f"Edge perturbato: {src_node} -> {dst_node}")

impacted_nodes = (diff.sum(dim=-1) > 1e-4).nonzero(as_tuple=True)[0].tolist()
print("Nodi impattati:", impacted_nodes)

Edge perturbato: 5 -> 0
Nodi impattati: [0, 43, 93]


In [7]:
edge_index = graph["edge_index"]
outgoing_from_0 = edge_index[1][edge_index[0] == 0].tolist()
print("Nodi raggiunti da edge uscenti da node0:", outgoing_from_0)
print("43 in outgoing?", 43 in outgoing_from_0)
print("93 in outgoing?", 93 in outgoing_from_0)

Nodi raggiunti da edge uscenti da node0: [43]
43 in outgoing? True
93 in outgoing? False


In [8]:
outgoing_from_43 = edge_index[1][edge_index[0] == 43].tolist()
print("Nodi raggiunti da edge uscenti da node43:", outgoing_from_43)
print("93 in outgoing?", 93 in outgoing_from_43)

Nodi raggiunti da edge uscenti da node43: [0, 93]
93 in outgoing? True
